# Darknode — agentic/business fine-tune (Qwen2.5-14B-Instruct, QLoRA)

Builds the flagship **Darknode** model: a strong open base tuned to be **agentic** (plans,
calls tools, completes multi-step tasks) and **business/enterprise-focused** — deliberately
trained away from casual daily-use chat. Method: 4-bit **QLoRA** via **Unsloth** (the practical
way to fit a 14B QLoRA on Kaggle's 15-16 GB GPU). Adapter is pushed to **`Darknod/Darknode`**.

### Why fine-tune, not from-scratch
Agentic ability only exists at scale; you can't train it from zero on Kaggle. Fine-tuning a
14B base specializes its behavior (tool-call format, enterprise tone, task focus) toward the
Darknode mission. This is the strongest agentic model reachable on this hardware.

### Setup before Run All
1. **Settings → Accelerator →** GPU `P100` (single 16 GB, simplest) or `T4 x2` (uses one T4).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets →** add `HF_TOKEN` (a Hugging Face **write** token). Pushes to `Darknod/Darknode`.
4. Run all. If the Unsloth install conflicts with Kaggle's preinstalled torch, run the install
   cell, then **Run → Restart & Run All** once.

In [ ]:
# Unsloth brings compatible torch/trl/peft/bitsandbytes. If this fights Kaggle's torch,
# restart the session after it finishes and Run All again.
!pip install -q unsloth
# Fallback (bleeding edge) if the line above errors on import later:
# !pip install -q --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" 

## 1. Configuration

In [ ]:
CONFIG = dict(
    base_model="unsloth/Qwen2.5-14B-Instruct-bnb-4bit",   # 4-bit Qwen2.5-14B-Instruct
    hf_repo="Darknod/Darknode",
    push_adapter=True,        # push the LoRA adapter (small, ~0.3-1 GB) at the end
    push_merged_16bit=False,  # also push a merged fp16 model (~28 GB upload — slow; off by default)
    push_gguf=False,          # also push a q4_k_m GGUF for llama.cpp/Ollama serving

    max_seq_length=1024,      # raise to 2048 on P100 if VRAM allows
    lora_r=16, lora_alpha=16, lora_dropout=0.0,

    batch=1, grad_accum=16,   # effective batch = batch*grad_accum sequences (batch=1 fits 14B on a 15GB T4)
    epochs=1, max_steps=500,  # max_steps>0 caps a session; set 0 to run a full epoch
    lr=2e-4, warmup=20, save_steps=100,

    max_examples=25000,       # cap the mixed dataset for a feasible Kaggle run
    include_xlam=True,        # Salesforce xLAM tool-calling (CC-BY-NC — research/non-commercial)

    out_dir="/kaggle/working/outputs", seed=3407,
)

# Injected as the system turn whenever a training example lacks one. This is what makes the
# model behave like an enterprise agent instead of a casual chatbot.
DARKNODE_SYSTEM = (
    "You are Darknode, an enterprise AI agent specialized in cybersecurity, software "
    "engineering, and business operations. You plan before acting, call tools precisely, and "
    "carry multi-step tasks to completion. You are professional, concise, and evidence-driven. "
    "You do not engage in casual small-talk or entertainment; you focus on getting business "
    "and technical work done correctly."
)

## 2. Load Qwen2.5-14B in 4-bit + attach LoRA (Unsloth)

In [ ]:
import os, torch, random, json
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    for _name in ("HF_TOKEN", "Darknode", "hf_token", "HF"):   # accept a few common secret labels
        try:
            HF_TOKEN = _us.get_secret(_name)
            if HF_TOKEN:
                print("using Kaggle secret:", _name); break
        except Exception:
            pass
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF token:", "yes" if HF_TOKEN else "no (adapter push will be skipped) — add a secret and enable Internet")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,            # auto: bf16 on new GPUs, fp16 on T4
    load_in_4bit=True,
    token=HF_TOKEN,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none", use_gradient_checkpointing="unsloth", random_state=CONFIG["seed"],
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
print("model + LoRA ready")

## 3. Build the agentic/business dataset
Mixes tool-calling, agent-trajectory, and cyber/business instruction sets. Each source loads
defensively (any failure is skipped), and a built-in seed set guarantees there is always data.
Every example is normalized to chat turns; a Darknode system prompt is injected where missing.

In [ ]:
from datasets import Dataset

def norm_role(r):
    r = (r or "").lower()
    if r in ("human", "user"): return "user"
    if r in ("gpt", "assistant", "model", "ai"): return "assistant"
    if r == "system": return "system"
    return "user"

def row_to_convo(row):
    conv = row.get("conversations") or row.get("conversation")
    if isinstance(conv, list) and conv and isinstance(conv[0], dict):
        out = []
        for t in conv:
            content = t.get("content") or t.get("value")
            if content: out.append({"role": norm_role(t.get("role") or t.get("from")), "content": str(content)})
        return out if len(out) >= 2 else None
    msgs = row.get("messages")
    if isinstance(msgs, list) and msgs and isinstance(msgs[0], dict) and "role" in msgs[0]:
        out = [{"role": norm_role(m.get("role")), "content": str(m.get("content", ""))} for m in msgs if m.get("content")]
        return out if len(out) >= 2 else None
    instr = row.get("instruction") or row.get("query") or row.get("prompt") or row.get("question")
    output = row.get("output") or row.get("response") or row.get("answer") or row.get("completion")
    if instr and output:
        convo = []
        if row.get("system"): convo.append({"role": "system", "content": str(row["system"])})
        user = str(instr) + (("\n\n" + str(row["input"])) if row.get("input") else "")
        convo.append({"role": "user", "content": user})
        convo.append({"role": "assistant", "content": str(output)})
        return convo
    return None

def adapt_xlam(row):
    q, tools, ans = row.get("query"), row.get("tools"), row.get("answers")
    if not (q and ans): return None
    sys = "You can call these tools (JSON schema):\n" + str(tools) + "\nReturn the required tool call(s) as a JSON list."
    return [{"role": "system", "content": sys}, {"role": "user", "content": str(q)}, {"role": "assistant", "content": str(ans)}]

DATASETS = [
    dict(repo="Salesforce/xlam-function-calling-60k", split="train", adapter="xlam", max=20000, enabled=CONFIG["include_xlam"], note="CC-BY-NC"),
    dict(repo="internlm/Agent-FLAN",                  split="train", adapter="auto", max=12000, enabled=True,  note="Apache-2.0"),
    dict(repo="trendmicro-ailab/Primus-Instruct",     split="train", adapter="auto", max=6000,  enabled=True,  note="MIT (cyber)"),
    dict(repo="glaiveai/glaive-function-calling-v2",  split="train", adapter="auto", max=8000,  enabled=False, note="string schema; off by default"),
]

SEED = [
    [{"role": "user", "content": "Our SIEM flagged 400 failed SSH logins from one IP in 5 minutes. What do you do?"},
     {"role": "assistant", "content": "Assessment: likely SSH brute-force. Steps: (1) block the source IP at the firewall/WAF; (2) check whether any of the targeted accounts succeeded afterward; (3) enforce key-based auth and disable password SSH; (4) enable fail2ban/rate-limiting; (5) file an incident ticket with the IP, timestamps, and targeted accounts. Want me to draft the firewall rule and the incident summary?"}],
    [{"role": "system", "content": "Available tool: create_incident(severity, service, summary)"},
     {"role": "user", "content": "Open a P1 incident for a payment-API outage."},
     {"role": "assistant", "content": "create_incident(severity=\"P1\", service=\"payment-api\", summary=\"Payment API outage — customers cannot complete checkout\")"}],
    [{"role": "user", "content": "Summarize this quarter's churn drivers for the exec review."},
     {"role": "assistant", "content": "I need the churn dataset or metrics to be accurate. Please share the cohort retention table or connect the analytics source; I'll return the top 3 drivers with supporting figures and a one-slide summary. I won't speculate on numbers I can't verify."}],
]

def load_one(spec):
    from datasets import load_dataset
    base = spec.get("split") or "train"
    ds = None
    try:
        ds = load_dataset(spec["repo"], split=base + "[:" + str(spec["max"]) + "]")
    except Exception:
        try:
            it = load_dataset(spec["repo"], split=base, streaming=True)
            rows = []
            for i, r in enumerate(it):
                rows.append(r)
                if i + 1 >= spec["max"]: break
            ds = Dataset.from_list(rows)
        except Exception as e:
            print("  skip", spec["repo"], "->", repr(e)[:130]); return []
    adapt = adapt_xlam if spec["adapter"] == "xlam" else row_to_convo
    out = []
    for r in ds:
        try:
            c = adapt(r)
            if c: out.append({"conversations": c})
        except Exception:
            pass
    print("  +", len(out), "from", spec["repo"], "(" + spec.get("note", "") + ")")
    return out

records = [{"conversations": c} for c in SEED]
for spec in DATASETS:
    if spec["enabled"]:
        records += load_one(spec)

# inject the Darknode system prompt where missing
for rec in records:
    if not (rec["conversations"] and rec["conversations"][0]["role"] == "system"):
        rec["conversations"] = [{"role": "system", "content": DARKNODE_SYSTEM}] + rec["conversations"]

random.Random(CONFIG["seed"]).shuffle(records)
records = records[:CONFIG["max_examples"]]
print("total training examples:", len(records))
raw = Dataset.from_list(records)

## 4. Render to the Qwen chat template

In [ ]:
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in batch["conversations"]]}

dataset = raw.map(to_text, batched=True, remove_columns=raw.column_names)
print(dataset)
print("\n--- sample ---\n", dataset[0]["text"][:800])

## 5. Train (QLoRA SFT via Unsloth + TRL)

In [ ]:
from trl import SFTTrainer, SFTConfig
bf16 = torch.cuda.is_bf16_supported()

args = SFTConfig(
    per_device_train_batch_size=CONFIG["batch"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    warmup_steps=CONFIG["warmup"],
    num_train_epochs=CONFIG["epochs"],
    max_steps=CONFIG["max_steps"] if CONFIG["max_steps"] else -1,
    learning_rate=CONFIG["lr"],
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=CONFIG["seed"],
    output_dir=CONFIG["out_dir"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    fp16=not bf16, bf16=bf16,
)

# If your TRL version rejects `tokenizer=`, change it to `processing_class=tokenizer`.
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset, args=args)

# resume from a prior session's checkpoint if one is present in output_dir
last_ckpt = None
if os.path.isdir(CONFIG["out_dir"]):
    cks = [d for d in os.listdir(CONFIG["out_dir"]) if d.startswith("checkpoint-")]
    if cks: last_ckpt = os.path.join(CONFIG["out_dir"], sorted(cks, key=lambda x: int(x.split("-")[-1]))[-1])
print("resuming from", last_ckpt) if last_ckpt else print("fresh training run")

stats = trainer.train(resume_from_checkpoint=last_ckpt)
print(stats)

## 6. Quick agentic smoke test

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": DARKNODE_SYSTEM},
    {"role": "user", "content": "A customer reports their API key was leaked in a public GitHub repo. Give me the response plan and the first tool call you would make."},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.4, do_sample=True, use_cache=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 7. Save + push to `Darknod/Darknode`

In [ ]:
os.makedirs("/kaggle/working/darknode-adapter", exist_ok=True)
model.save_pretrained("/kaggle/working/darknode-adapter")
tokenizer.save_pretrained("/kaggle/working/darknode-adapter")
print("adapter saved locally")

if HF_TOKEN:
    if CONFIG["push_adapter"]:
        model.push_to_hub(CONFIG["hf_repo"], token=HF_TOKEN)
        tokenizer.push_to_hub(CONFIG["hf_repo"], token=HF_TOKEN)
        print("adapter -> https://huggingface.co/" + CONFIG["hf_repo"])
    if CONFIG["push_merged_16bit"]:
        model.push_to_hub_merged(CONFIG["hf_repo"], tokenizer, save_method="merged_16bit", token=HF_TOKEN)
        print("merged fp16 pushed")
    if CONFIG["push_gguf"]:
        model.push_to_hub_gguf(CONFIG["hf_repo"], tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
        print("gguf pushed")
else:
    print("no HF_TOKEN — nothing pushed. Adapter is in /kaggle/working/darknode-adapter")

## Continue, serve, scale
- **Keep training:** re-run in a new session; it resumes from the pushed checkpoint / adapter.
  Raise `max_steps` (or set it to 0 for a full epoch) and `max_examples` as your GPU budget allows.
- **Serve:** load the base model + this adapter with PEFT, or push a GGUF (`push_gguf=True`) and
  run it in Ollama / llama.cpp — which can back the Darknode web app's AI endpoints.
- **Bigger / stronger:** the ceiling here is the 14B base. For more, you'd move off Kaggle to a
  rented A100/H100 (more context, full-parameter SFT, or a larger base).
- **Licensing:** xLAM is CC-BY-NC (research/non-commercial). For a commercial Darknode model set
  `include_xlam=False` and rely on the Apache/MIT sources (Agent-FLAN, Primus, glaive).